In [1]:
import json
import faiss
import requests
import json
import config
import time
import unicodedata
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

In [2]:
def load_chunks_json(file_path):
    chunks = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))
    return chunks

In [119]:
all_chunks = load_chunks_json("./data/chunked_data/chunks_v2_400docs.json")

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [chunk["text"] for chunk in all_chunks]

embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

KeyboardInterrupt: 

In [3]:
# np.save("data/embeddings/embeddings_v2_400docs.npy", embeddings)
all_chunks = load_chunks_json("./data/chunked_data/chunks_v2_400docs.json")
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = np.load("data/embeddings/embeddings_v2_400docs.npy")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
dim = embeddings.shape[1]

index = faiss.IndexFlatL2(dim)
index.add(embeddings)

In [5]:
def retrieve(query, k=5):
    q_emb = model.encode([query]).astype("float32")
    
    distances, indices = index.search(q_emb, k)
    
    results = [all_chunks[i] for i in indices[0]]
    return results

In [ ]:
results = retrieve("What is the role of circumstantial evidence in murder cases?", k=5)

for r in results[:2]:
    print(r["id"])
    print(r["text"])
    print("="*50)
    print("\n")

Abdul Nassar vs The State Of Kerala_7 January 2025_23

Case: Abdul Nassar vs The State Of Kerala
Date: 7 January 2025

and defence witness must be meticulously discussed and analysed. Each witness's evidence should be assessed in its entirety to ensure no material aspect is overlooked. (ii). Circumstantial evidence is evidence that relies on an inference to connect it to a conclusion of fact. Thus, the reasonable inferences that can be drawn from the testimony of each witness must be explicitly delineated. (iii). Each of the links of incriminating circumstantial evidence should be meticulously examined so as to find out if each one of the circumstances is proved individually and whether collectively taken, they forge an unbroken chain consistent only with the hypothesis of the guilt of the accused and totally inconsistent with his innocence. (iv). The judgment must comprehensively elucidate the rationale for accepting or rejecting specific pieces of evidence, demonstrating how the conc

In [6]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, retrieved_chunks, top_k=5):
    pairs = [(query, chunk["text"]) for chunk in retrieved_chunks]
    
    scores = reranker.predict(pairs)
    
    ranked = sorted(zip(scores, retrieved_chunks), key=lambda x: x[0], reverse=True)
    
    return [chunk for _, chunk in ranked[:top_k]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def retrieve_with_rerank(query):
    initial = retrieve(query, k=20)
    final = rerank(query, initial, top_k=5)
    return final

In [ ]:
results = retrieve_with_rerank("delay in FIR criminal case India")

for r in results[:2]:
    print(r["case_name"])
    print(r["text"])
    print("="*50)

Firoz Khan Akbarkhan vs The State Of Maharashtra

Case: Firoz Khan Akbarkhan vs The State Of Maharashtra
Date: 24 March 2025

the inordinate delays therein could not be sufficiently explained. Delay of about 27 days, in a case where communal violence had broken out, was held not fatal, in Lal Bahadur v State (NCT of Delhi), (2013) 4 SCC 557. Delay of over 2 years in recording witness statements was deemed not fatal, when explained, in Baldev Singh v State of Punjab, CRIMINAL APPEAL NO.257 OF 2013 16 of 27 (2014) 12 SCC 473. Delay in recording witness statements was held not fatal per se in Sunil Kumar v State of Rajasthan, (2005) 9 SCC 283 and V K Mishra v State of Uttarakhand, (2015) 9 SCC 588. Delay in recording statements of witnesses was held to have cast serious doubts on the prosecution version in Shahid Khan v State of Rajasthan, (2016) 4 SCC 96 and Jafarudheen v State of Kerala, (2022) 8 SCC 440.   7 was held, in Goutam Joardar v State of W. B., (2022) 17 SCC 549, by a Coordina

In [8]:
def build_context(chunks):
    context = ""
    for i, c in enumerate(chunks):
        context += f"[Source {i+1}]\n{c['text']}\n\n"
    return context

In [9]:
def build_prompt(query, context):
    return f"""
You are a legal assistant.

Answer the question using ONLY the context provided below.

Context:
{context}

Question:
{query}

Instructions:
- Be concise and precise
- Use only the provided context
- Always cite sources inline strictly in this format: [Amit Kumar vs Union of India, 24 Mar 2025]. 
- Use nuanced legal language (avoid absolute terms)
- Do NOT use "supra", "ibid", or shortened references.
- If context is partially sufficient, answer what is supported and briefly note missing aspects
- Do NOT introduce new case names or legal provisions not present in the context.
- Use multiple cases to support reasoning where possible, not just one dominant case.

Structure:
1. Legal Principle (3-5 lines)
2. Key Reasoning from Cases (3-5 lines, with citations)
3. Conclusion (short)

At the end, provide:
Confidence: High / Medium / Low based on completeness of context

Answer:
"""

In [ ]:
def call_ollama_local(prompt, model="llama3"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    
    return response.json()["response"]

from ollama import Client

client = Client(
    host='https://ollama.com',
    headers={'Authorization': 'Bearer ' + config.ollama_api_key}
)

def call_ollama_cloud(prompt, model="gpt-oss:120b"):
    response = client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response['message']['content']

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = text.replace("\u200b", "")   # zero-width space
    text = text.replace("\u202f", " ")  # narrow no-break space
    
    return text

def generate_answer(query, chunks, model="gpt-oss:120b"):
    
    if not chunks:
        return "No relevant legal context found to answer the query."
    
    context = ""
    for c in chunks:
        context += (
            f"{c['text']}\n\n"
        )

    prompt = build_prompt(query, context)
    answer = call_ollama_cloud(prompt, model)
    answer_clean = normalize_text(answer)

    return answer_clean

In [11]:
def rewrite_query(query):
    prompt = f"""
You are a classifier and rewriter.

Task:
1. Determine if the query is related to law, legal systems, courts, crime, rights, or legal procedures.
2. If YES:
   - Rewrite it into a clear and concise legal query (Indian legal context if applicable).
   - Keep it short (1 line).
3. If NO:
   - Return exactly: Not a legal query

Rules:
- Be permissive: if the query could reasonably relate to law or legal concepts, treat it as legal.
- Queries about crime, police, FIR, courts, evidence, rights, or procedures are ALL legal.
- Do NOT overthink or reject borderline queries.

Query:
{query}

Output:
"""
    return call_ollama_cloud(prompt)

def diversify_chunks(chunks):
    seen_cases = dict()
    unique = []

    for c in chunks:
        if c["case_name"] not in seen_cases:
            unique.append(c)
            seen_cases[c["case_name"]] = 1
        else:
            seen_cases[c["case_name"]] += 1
        if seen_cases[c["case_name"]] <= 3:
            unique.append(c)

    return unique

def merge_same_case(chunks):
    merged = {}
    
    for c in chunks:
        key = c["case_name"]
        if key not in merged:
            merged[key] = c
        else:
            merged[key]["text"] += " " + c["text"]
    
    return list(merged.values())

In [128]:
query = "Does Article 21 guarantee anticipatory bail?"

rewritten_query = rewrite_query(query)
if rewritten_query.strip() == "Not a legal query":
    print("The query is not related to Indian Legal system.")
else:
    retrieved_chunks = retrieve_with_rerank(rewritten_query)
    modified_retrieved_chunks = merge_same_case(retrieved_chunks)
    if len(modified_retrieved_chunks) == 0:
        print("No relevant legal context found for your query.")
    answer = generate_answer(rewritten_query, modified_retrieved_chunks)
    print(answer)

**Legal Principle**  
Article 21 guarantees the right to life and personal liberty, but the entitlement to anticipatory bail is not deemed an inherent facet of that constitutional right; it is a statutory remedy created by legislation after the Constitution came into force.

**Key Reasoning from Cases**  
The Supreme Court in *Devinder Kumar Bansal vs The State Of Punjab* observed that anticipatory bail “is essentially a statutory right … It cannot be considered as an essential ingredient of Article 21 of the Constitution” and may be granted only in “very exceptional cases” where the applicant is prima facie falsely implicated or the allegations are frivolous or politically motivated [Devinder Kumar Bansal vs The State Of Punjab, 3 March 2025]. Other judgments, such as *Central Bureau of Investigation v. V. Vijay Sai Reddy*, stress the need to satisfy “reasonable grounds for believing” a genuine case exists before bail is awarded, underscoring that anticipatory bail remains a discretio

In [12]:
query = "Does Article 21 guarantee anticipatory bail?"

rewritten_query = rewrite_query(query)
if rewritten_query.strip() == "Not a legal query":
    print("The query is not related to Indian Legal system.")
else:
    retrieved_chunks = retrieve_with_rerank(rewritten_query)
    modified_retrieved_chunks = merge_same_case(retrieved_chunks)
    if len(modified_retrieved_chunks) == 0:
        print("No relevant legal context found for your query.")
    answer = generate_answer(rewritten_query, [])
    print(answer)

**Legal Principle**  
Article 21 guarantees the protection of life and personal liberty, but the extent to which it embraces the right to anticipatory bail is not definitively settled in the supplied material.

**Key Reasoning from Cases**  
The provided excerpts do not contain any judicial pronouncements expressly linking Article 21 to an inherent right to anticipatory bail, nor do they cite decisions interpreting Article 21 in that specific context. Consequently, no case law is available in the material to support a conclusion that anticipatory bail is a facet of the right under Article 21.

**Conclusion**  
Based on the current context, it cannot be affirmed that Article 21 guarantees the right to anticipatory bail; the material lacks the requisite authoritative statements on this point.

**Confidence:** Low – the context does not supply the needed judicial analysis or citations.


In [ ]:
evaluation_set = [
    # ✅ FIR / Criminal Procedure (should work well)
    {
        "query": "What is the effect of delay in FIR?",
        "type": "in_domain",
        "expected_keywords": ["delay", "credibility", "doubt"]
    },
    {
        "query": "When can investigation be transferred to CBI?",
        "type": "in_domain",
        "expected_keywords": ["exceptional", "not routine", "CBI"]
    },
    {
        "query": "Can High Court order CBI investigation?",
        "type": "in_domain",
        "expected_keywords": ["constitutional", "power", "exceptional"]
    },
    {
        "query": "When should courts avoid transferring investigation to CBI?",
        "type": "in_domain",
        "expected_keywords": ["not routine", "exceptional", "vague allegations"]
    },
    {
        "query": "Can courts grant relief after delay in exercising legal rights?",
        "type": "in_domain",
        "expected_keywords": ["delay", "diligence", "no legal right"]
    },
    {
        "query": "When can writ petition be dismissed due to lack of legal right?",
        "type": "in_domain",
        "expected_keywords": ["legal right", "mandamus", "maintainable"]
    },
    # ⚠️ Edge legal queries (partial expected)
    {
        "query": "What is anticipatory bail?",
        "type": "out_of_context",
        "expected_behavior": "partial_or_refuse"
    },
    {
        "query": "What is Section 482 CrPC?",
        "type": "out_of_context",
        "expected_behavior": "partial_or_refuse"
    },
    # ❌ Out of domain (must reject)
    {
        "query": "What is reinforcement learning?",
        "type": "out_of_domain",
        "expected_behavior": "reject"
    },
    {
        "query": "Explain neural networks",
        "type": "out_of_domain",
        "expected_behavior": "reject"
    },
    # 🔥 Hallucination traps
    {
        "query": "Is anticipatory bail a fundamental right under Article 21?",
        "type": "trap",
        "expected_behavior": "no_hallucination"
    },
    {
        "query": "Does every delay in FIR make prosecution invalid?",
        "type": "trap",
        "expected_behavior": "nuanced"
    },
]

In [146]:
import re

def is_citation_in_context(citation, context):
    words = citation.lower().split()
    
    match_count = sum(1 for w in words if w in context)
    
    return match_count >= max(2, len(words) // 2)

def classify_citations(answer, chunks):
    context = " ".join([c["text"] for c in chunks]).lower()
    case_names = [c["case_name"].lower() for c in chunks]

    cited = re.findall(r'\[(.*?)\]', answer)
    print("Cited cases:", cited)

    primary, secondary, hallucinated = 0, 0, 0

    for c in cited:
        c_low = c.lower()

        if any(name in c_low for name in case_names):
            primary += 1
        elif c_low in context:
            secondary += 1
        else:
            hallucinated += 1

    return primary, secondary, hallucinated

def evaluate_single(query, item, answer, chunks):
    # ---- Build context ----
    context = " ".join([c["text"] for c in chunks]).lower()
    case_names = [c["case_name"].lower() for c in chunks]
    print("Case names in context:", case_names)

    # ---- Extract citations ----
    cited = set(re.findall(r'\[\s*(.*?)\s*\]', answer))
    print("Cited cases:", cited)

    primary, secondary, hallucinated = 0, 0, 0

    for c in cited:
        c_low = c.lower()

        if any(name in c_low for name in case_names):
            primary += 1
        elif is_citation_in_context(c_low, context):
            secondary += 1
        else:
            hallucinated += 1

    total = len(cited)

    if total > 0:
        primary_score = primary / total
        secondary_score = secondary / total
        hallucination_rate = hallucinated / total
    else:
        primary_score = 0
        secondary_score = 0
        hallucination_rate = 0

    # ---- LLM evaluation ----
    try:
        llm_eval_raw = llm_evaluate(query, context, answer)
        llm_eval = json.loads(llm_eval_raw)
    except:
        llm_eval = {"error": "invalid_json"}

    return {
        "query": query,

        # ---- LLM judgment ----
        "llm_eval": llm_eval,

        # ---- Citation metrics ----
        "num_citations": total,
        "primary_citation_score": primary_score,
        "secondary_citation_score": secondary_score,
        "hallucination_rate": hallucination_rate,

        # ---- Debugging ----
        "citations_extracted": list(cited),
        "answer_preview": answer[:200]
    }

def summarize_results(results):

    total = len(results)

    # ---- Aggregate metrics ----
    avg_primary = sum(r["primary_citation_score"] for r in results) / total
    avg_secondary = sum(r["secondary_citation_score"] for r in results) / total
    avg_hallucination = sum(r["hallucination_rate"] for r in results) / total
    avg_keyword = sum(r["keyword_score"] for r in results) / total

    total_citations = sum(r["num_citations"] for r in results)

    # ---- Rejection accuracy ----
    out_of_domain = [r for r in results if r["type"] == "out_of_domain"]
    if out_of_domain:
        rejection_accuracy = sum(r["correct_rejection"] for r in out_of_domain) / len(out_of_domain)
    else:
        rejection_accuracy = 0

    # ---- Breakdown by type ----
    def avg_metric(group, key):
        return sum(r[key] for r in group) / len(group) if group else 0

    in_domain = [r for r in results if r["type"] == "in_domain"]
    out_of_context = [r for r in results if r["type"] == "out_of_context"]
    traps = [r for r in results if r["type"] == "trap"]

    print("\n========== RAG EVALUATION SUMMARY ==========\n")

    print(f"Total Queries: {total}")
    print(f"Total Citations: {total_citations}")

    print("\n--- Overall Metrics ---")
    print(f"Primary Citation Score: {avg_primary:.2f}")
    print(f"Secondary Citation Score: {avg_secondary:.2f}")
    print(f"Hallucination Rate: {avg_hallucination:.2f}")
    print(f"Keyword Relevance Score: {avg_keyword:.2f}")
    print(f"Rejection Accuracy (OOD): {rejection_accuracy:.2f}")

    print("\n--- In-Domain Performance ---")
    print(f"Primary: {avg_metric(in_domain, 'primary_citation_score'):.2f}")
    print(f"Hallucination: {avg_metric(in_domain, 'hallucination_rate'):.2f}")
    print(f"Keyword: {avg_metric(in_domain, 'keyword_score'):.2f}")

    print("\n--- Out-of-Context Performance ---")
    print(f"Primary: {avg_metric(out_of_context, 'primary_citation_score'):.2f}")
    print(f"Hallucination: {avg_metric(out_of_context, 'hallucination_rate'):.2f}")

    print("\n--- Trap Queries (Hallucination Check) ---")
    print(f"Hallucination Rate: {avg_metric(traps, 'hallucination_rate'):.2f}")

    print("\n===========================================\n")

def llm_evaluate(query, context, answer):

    prompt = f"""
    You are an expert legal evaluator.

    Evaluate strictly using ONLY the allowed values based on the provided context and the answer.

    Grounding:
    - Allowed values: Yes / Partial / No

    Completeness:
    - Allowed values: Complete / Partial / Incomplete

    Hallucination:
    - Allowed values: None / Minor / Major

    Return EXACTLY this format:
    {{
    "grounding": "Yes/Partial/No",
    "completeness": "Complete/Partial/Incomplete",
    "hallucination": "None/Minor/Major",
    "score": 1-5
    }}

    Question:
    {query}

    Context:
    {context}

    Answer:
    {answer}
    """

    response = call_ollama_cloud(prompt)

    return response

def run_full_evaluation(evaluation_set):

    results = []

    for item in tqdm(evaluation_set):
        query = item["query"]

        # ---- Step 1: Rewrite / classify ----
        rewritten_query = rewrite_query(query)

        # ---- Step 2: Handle rejection ----
        if rewritten_query.strip() == "Not a legal query":
            answer = "REJECTED"
            chunks = []

        else:
            # ---- Step 3: Retrieval ----
            chunks = retrieve_with_rerank(rewritten_query)

            # ---- Step 4: Post-process ----
            chunks = merge_same_case(chunks)

            # ---- Step 5: Generation ----
            answer = generate_answer(rewritten_query, chunks)

        # ---- Step 6: Evaluation ----
        result = evaluate_single(query, item, answer, chunks)
        results.append(result)

    return results

In [147]:
results = run_full_evaluation(evaluation_set[:1])

  0%|          | 0/1 [00:00<?, ?it/s]

Case names in context: ['the state of himachal pradesh vs rajesh kumar munnu', 'inder singh vs the state of madhya pradesh', 'a rajendra vs gonugunta madhusudhan rao', 'firoz khan akbarkhan vs the state of maharashtra']
Cited cases: {'State Of Himachal Pradesh vs Rajesh Kumar Munnu, 20 February 2025'}


100%|██████████| 1/1 [00:18<00:00, 18.85s/it]


In [125]:
print(results[0]['answer'])

KeyError: 'answer'

In [148]:
results

[{'query': 'What is the effect of delay in FIR?',
  'llm_eval': {'grounding': 'Yes',
   'completeness': 'Complete',
   'hallucination': 'None',
   'score': 5},
  'num_citations': 1,
  'primary_citation_score': 0.0,
  'secondary_citation_score': 1.0,
  'hallucination_rate': 0.0,
  'citations_extracted': ['State Of Himachal Pradesh vs Rajesh Kumar Munnu, 20 February 2025'],
  'answer_preview': '**1. Legal Principle**  \nAn unexplained or unjustified delay in lodging an FIR is ordinarily considered detrimental to the prosecution’s case and may confer a statutory benefit on the accused, potenti'}]